# Calibration protocol для обучающей и бенчмарк выборок из датасета BigP300BCI

# BigP3BCI Calibration Splits

Этот ноутбук использовался для построения calibration/test split'ов для downstream-экспериментов на BigP3BCI.

Основные этапы:
1. Nested stratified sampling
2. Построение split для одного субъекта
3. Построение split'ов для всех субъектов
4. Подсчёт статистик нормализации

Выходные данные сохранены в:
- `data/processed/bigp3bci/downstream/train/splits/`
- `data/processed/bigp3bci/downstream/benchmark/splits/`
- `data/processed/bigp3bci/downstream/train/stats/`
- `data/processed/bigp3bci/downstream/benchmark/stats/`

## Конфиг

In [ ]:
from pathlib import Path
import json
import numpy as np

# --- Paths ---
ROOT = Path().resolve().parents[1]  # корень проекта

DATA_ROOT =  ROOT / "data" / "processed" / "bigp3bci"
DOWN_ROOT = DATA_ROOT / "downstream"

TRAIN_ROOT = DOWN_ROOT / "train"
BENCH_ROOT = DOWN_ROOT / "benchmark"

SPLITS_TRAIN = TRAIN_ROOT / "splits"
SPLITS_BENCH = BENCH_ROOT / "splits"

SPLITS_TRAIN.mkdir(parents=True, exist_ok=True)
SPLITS_BENCH.mkdir(parents=True, exist_ok=True)

# --- Protocol ---
CALIB_POOL_RATIO = 0.70
P_LIST = [10, 20, 40, 60, 100]

GLOBAL_SEED = 2026 
print("TRAIN_ROOT:", TRAIN_ROOT)
print("BENCH_ROOT:", BENCH_ROOT)
print("SPLITS_TRAIN:", SPLITS_TRAIN)
print("SPLITS_BENCH:", SPLITS_BENCH)
print("Protocol:", {"calib_pool_ratio": CALIB_POOL_RATIO, "p_list": P_LIST, "seed": GLOBAL_SEED})

TRAIN_ROOT: C:\Users\Таисия\Desktop\МФТИ\Диплом_BCI\Выборки\BigP300BCI\downstream\train
BENCH_ROOT: C:\Users\Таисия\Desktop\МФТИ\Диплом_BCI\Выборки\BigP300BCI\downstream\benchmark
SPLITS_TRAIN: C:\Users\Таисия\Desktop\МФТИ\Диплом_BCI\Выборки\BigP300BCI\downstream\splits\train
SPLITS_BENCH: C:\Users\Таисия\Desktop\МФТИ\Диплом_BCI\Выборки\BigP300BCI\downstream\splits\benchmark
Protocol: {'calib_pool_ratio': 0.7, 'p_list': [10, 20, 40, 60, 100], 'seed': 2026}


In [2]:
# Проверка
train_files = sorted(TRAIN_ROOT.glob("*.npz"))
bench_files = sorted(BENCH_ROOT.glob("*.npz"))
len(train_files), len(bench_files), train_files[:2], bench_files[:2]

(93,
 65,
 [WindowsPath('C:/Users/Таисия/Desktop/МФТИ/Диплом_BCI/Выборки/BigP300BCI/downstream/train/subj_000.npz'),
  WindowsPath('C:/Users/Таисия/Desktop/МФТИ/Диплом_BCI/Выборки/BigP300BCI/downstream/train/subj_001.npz')],
 [WindowsPath('C:/Users/Таисия/Desktop/МФТИ/Диплом_BCI/Выборки/BigP300BCI/downstream/benchmark/subj_051.npz'),
  WindowsPath('C:/Users/Таисия/Desktop/МФТИ/Диплом_BCI/Выборки/BigP300BCI/downstream/benchmark/subj_052.npz')])

## Nested stratified sampling

In [3]:
def make_nested_stratified_indices(y_pool, p_list, seed):
    """
    y_pool: (N,) labels within calib_pool
    returns dict {p: idx_in_pool}
    """
    
    rng = np.random.default_rng(seed)
    
    y_pool = np.asarray(y_pool)
    
    # --- split indices by class ---
    idx0 = np.where(y_pool == 0)[0]
    idx1 = np.where(y_pool == 1)[0]
    
    # --- deterministic shuffle ---
    rng.shuffle(idx0)
    rng.shuffle(idx1)
    
    n0 = len(idx0)
    n1 = len(idx1)
    N = len(y_pool)
    
    nested = {}
    
    for p in sorted(p_list):
        frac = p / 100.0
        
        k0 = int(round(frac * n0))
        k1 = int(round(frac * n1))
        
        sel0 = idx0[:k0]
        sel1 = idx1[:k1]
        
        idx = np.concatenate([sel0, sel1])
        idx.sort()
        
        nested[p] = idx
    
    return nested

In [4]:
# Проверка
import numpy as np

d = np.load(train_files[0])
y = d["y"]

# pool
N = len(y)
pool = int(np.ceil(CALIB_POOL_RATIO * N))
y_pool = y[:pool]

nested = make_nested_stratified_indices(y_pool, P_LIST, GLOBAL_SEED)

for p, idx in nested.items():
    print(p, len(idx), float((y_pool[idx]==1).mean()))

10 1809 0.09286898839137644
20 3619 0.09311964631113567
40 7238 0.09311964631113567
60 10856 0.09303610906411201
100 18094 0.09306952580966066


In [5]:
set(nested[10]).issubset(set(nested[20]))

True

## Построение split одного субъекта

In [6]:
def build_subject_splits(npz_path, out_root, seed):
    
    d = np.load(npz_path, allow_pickle=True)
    
    y = d["y"]
    N = len(y)
    
    # --- temporal split ---
    pool_size = int(np.ceil(CALIB_POOL_RATIO * N))
    
    calib_pool_idx = np.arange(pool_size)
    test_rest_idx = np.arange(pool_size, N)
    
    # --- nested calibration ---
    nested_pool = make_nested_stratified_indices(
        y[calib_pool_idx],
        P_LIST,
        seed
    )
    
    # map to global indices
    nested_global = {
        str(p): calib_pool_idx[idx].tolist()
        for p, idx in nested_pool.items()
    }
    
    # --- metadata ---
    out = {
        "calib_pool_idx": calib_pool_idx.tolist(),
        "test_rest_idx": test_rest_idx.tolist(),
        "calib_idx": nested_global,
        "seed": seed,
        "sizes": {
            "N_total": int(N),
            "N_pool": int(len(calib_pool_idx)),
            "N_test": int(len(test_rest_idx))
        },
        "class_counts": {
            "pool": int(np.sum(y[calib_pool_idx])),
            "test": int(np.sum(y[test_rest_idx]))
        }
    }
    
    sid = npz_path.stem
    out_path = out_root / f"{sid}.json"
    
    with open(out_path, "w") as f:
        json.dump(out, f, indent=2)
    
    return out

In [7]:
# Проверка
test_out = build_subject_splits(train_files[0], SPLITS_TRAIN, GLOBAL_SEED)

test_out["sizes"]

{'N_total': 25848, 'N_pool': 18094, 'N_test': 7754}

In [8]:
# Проверка
all(i < test_out["sizes"]["N_pool"] for i in test_out["calib_pool_idx"][:10])

True

In [9]:
# Проверка
pool_set = set(test_out["calib_pool_idx"])
p10_set = set(test_out["calib_idx"]["10"])
p10_set.issubset(pool_set)

True

## BUILD ALL SPLITS

In [10]:
from tqdm.auto import tqdm

# --- Train ---
for f in tqdm(train_files):
    build_subject_splits(f, SPLITS_TRAIN, GLOBAL_SEED)

# --- Benchmark ---
for f in tqdm(bench_files):
    build_subject_splits(f, SPLITS_BENCH, GLOBAL_SEED)

  0%|          | 0/93 [00:00<?, ?it/s]

  0%|          | 0/65 [00:00<?, ?it/s]

## Подсчёт статистики

### Пути

In [ ]:
STATS_TRAIN = TRAIN_ROOT / "stats"
STATS_BENCH = BENCH_ROOT / "stats"

STATS_TRAIN.mkdir(parents=True, exist_ok=True)
STATS_BENCH.mkdir(parents=True, exist_ok=True)

STATS_TRAIN, STATS_BENCH

(WindowsPath('C:/Users/Таисия/Desktop/МФТИ/Диплом_BCI/Выборки/BigP300BCI/downstream/stats/train'),
 WindowsPath('C:/Users/Таисия/Desktop/МФТИ/Диплом_BCI/Выборки/BigP300BCI/downstream/stats/benchmark'))

### Функция вычисления stats

In [12]:
def compute_stats(X, idx, L_raw):
    """
    X: (N,C,L)
    idx: calibration indices
    """
    X_cal = X[idx, :, :L_raw]  # remove padding
    
    mean = X_cal.mean(axis=(0,2))
    std = X_cal.std(axis=(0,2)) + 1e-8
    
    return mean.astype(np.float32), std.astype(np.float32)

### Build stats одного субъекта

In [13]:
def build_subject_stats(npz_path, split_root, out_root):
    
    sid = npz_path.stem
    
    # --- load data ---
    d = np.load(npz_path, allow_pickle=True)
    X = d["X"]
    L_raw = int(d["L_raw"])
    
    # --- load splits ---
    split_path = split_root / f"{sid}.json"
    with open(split_path, "r") as f:
        split = json.load(f)
    
    # --- compute per p ---
    for p, idx in split["calib_idx"].items():
        
        idx = np.array(idx)
        
        mean, std = compute_stats(X, idx, L_raw)
        
        out_path = out_root / f"{sid}_p{p}.npz"
        
        np.savez(
            out_path,
            mean=mean,
            std=std
        )

### тест одного субъекта

In [14]:
build_subject_stats(train_files[0], SPLITS_TRAIN, STATS_TRAIN)

list(STATS_TRAIN.glob("*p10.npz"))[:3]

[WindowsPath('C:/Users/Таисия/Desktop/МФТИ/Диплом_BCI/Выборки/BigP300BCI/downstream/stats/train/subj_000_p10.npz')]

### Массовый прогон

In [15]:
# --- Train ---
for f in train_files:
    build_subject_stats(f, SPLITS_TRAIN, STATS_TRAIN)

# --- Benchmark ---
for f in bench_files:
    build_subject_stats(f, SPLITS_BENCH, STATS_BENCH)

In [16]:
# Проверка
d = np.load(list(STATS_TRAIN.glob("*p10.npz"))[0])
d["mean"].shape, d["std"].shape

((14,), (14,))

In [17]:
# Проверка
d["mean"][:5], d["std"][:5]

(array([ 2.8593652e-08,  4.5065840e-09, -1.6536484e-08, -2.9537018e-08,
        -1.2835718e-08], dtype=float32),
 array([9.1436104e-06, 8.0176169e-06, 6.7256910e-06, 5.9277681e-06,
        6.0463576e-06], dtype=float32))